# DINO CLS-Embedding Displacement Predictor
Given two 224×224 RGB frames, predict the (Δx, Δy, Δz) robot arm displacement in mm.

**Data**: HDF5 files `data/robot_*.hdf5`
- `frame_i_x` — `(224, 224, 3)` uint8 BGR frame
- `frame_i_y` — `(3,)` float32 motor position `[x, y, z]` in mm

**Target**: `delta_xyz = xyz[j] − xyz[i]` — net 3D displacement between two frames

**Architecture** (only the last transformer block + head are trainable — everything before it stays frozen):
```
frame_i ─ DINOv3 ViT-S+/16 blocks 0-10 (frozen) ─ block 11 + final norm (trainable) ─ CLS token emb_a (384,)
frame_j ─ DINOv3 ViT-S+/16 blocks 0-10 (frozen) ─ block 11 + final norm (trainable) ─ CLS token emb_b (384,)

head_input   = [emb_a, emb_b, emb_a − emb_b]            (1152,)
displacement = MLP(head_input) → (3,)
```
ViT-S+/16 (`vit_small_plus_patch16_dinov3.lvd1689m`) natively produces a 384-dim pooled/CLS
embedding via `forward_head(..., pre_logits=True)` — unlike ConvNeXt-Tiny (whose only native
pooled output is 768-dim, from its final stage), no manual pooling of an intermediate layer is
needed here.

Only the last transformer block (`blocks.11`) and the final norm are unfrozen — this lets the
encoder reshape the features that actually feed the embedding for this displacement task, without
backpropagating through (and risking destroying) the lower-level, general-purpose features in the
earlier frozen blocks, and it's far cheaper than fine-tuning the whole network. The encoder's
unfrozen params and the head are optimized with separate learning rates (encoder much lower, since
it starts from a strong pretrained init and can be destroyed by a large LR).

**SigReg (variance regularization)**: fine-tuning part of the encoder end-to-end on a narrow
regression target risks *representation collapse* — the encoder could shrink all embeddings toward
a single point, since a constant embedding can still minimize the primary regression loss in
combination with the head. To prevent this, a variance/"sigreg" hinge loss is added: for each
embedding dimension, compute the standard deviation *across the batch* and penalize any dimension
whose std falls below a target `γ`. This keeps the embedding space spread out and informative,
independent of the regression loss.

**Loss**: `Huber(pred, target) + λ · SigReg(emb_a, emb_b)`.

### Imports

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
! pip install -q timm h5py

In [ ]:
import h5py
import multiprocessing
import warnings
import numpy as np
import timm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

multiprocessing.set_start_method('fork', force=True)
warnings.filterwarnings('ignore')

# ── paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'model_notebooks' else Path.cwd()
CKPT_DIR = REPO_ROOT / 'checkpoints_dino_disp'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')


## Data
### Load episode positions

In [ ]:
HDF5_DIR   = REPO_ROOT / 'data'
HDF5_FILES = sorted(HDF5_DIR.glob('robot_*.hdf5'))
assert HDF5_FILES, f'No robot_*.hdf5 files found in {HDF5_DIR}'
assert len(HDF5_FILES) >= 2, 'Need at least 2 hdf5 files to hold one out for validation'

IMG_SIZE = 224

# Hold out the last file (by sorted filename) entirely as a validation episode —
# its frames are never sampled during training.
TRAIN_FILES = HDF5_FILES[:-1]
VAL_FILES   = HDF5_FILES[-1:]
print(f'Train files: {[p.name for p in TRAIN_FILES]}')
print(f'Val file   : {[p.name for p in VAL_FILES]}')


def load_episode_data(files):
    episode_data = []
    for path in files:
        with h5py.File(path, 'r') as f:
            n = sum(1 for k in f.keys() if k.endswith('_x'))
            xyz = torch.from_numpy(
                np.stack([f[f'frame_{i}_y'][:] for i in range(n)]).astype(np.float32)
            )  # (n, 3)
        episode_data.append({'path': str(path), 'n': n, 'xyz': xyz})
        print(f'{path.name}: {n} frames  xyz range [{xyz.min():.1f}, {xyz.max():.1f}] mm')
    return episode_data


print('\nTrain episodes:')
episode_data = load_episode_data(TRAIN_FILES)
print('\nVal episodes:')
val_episode_data = load_episode_data(VAL_FILES)

print(f'\nTrain total : {sum(ep["n"] for ep in episode_data):,} frames')
print(f'Val total   : {sum(ep["n"] for ep in val_episode_data):,} frames')


### Dataset
Each item samples two random frames `(i, j)` from the **same episode** and returns:
- `frame_i`, `frame_j`: `(3, H, W)` uint8
- `delta_xyz`: `xyz[j] − xyz[i]` in mm — 3D displacement to predict

In [ ]:
class PairDisplacementDataset(Dataset):
    """Random pairs (frame_i, frame_j) from the same episode.

    Target: delta_xyz = xyz[j] - xyz[i] in mm.
    Frames are lazy-loaded from HDF5 per __getitem__.
    """

    def __init__(self, episode_data):
        self.episodes = episode_data
        self.ep_ends  = []
        total = 0
        for ep in episode_data:
            total += ep['n']
            self.ep_ends.append(total)
        self.total = total

    def __len__(self):
        return self.total

    def _ep_of(self, idx):
        for k, end in enumerate(self.ep_ends):
            if idx < end:
                return k
        return len(self.ep_ends) - 1

    def _read_frame(self, path, frame_idx):
        with h5py.File(path, 'r') as f:
            bgr = f[f'frame_{frame_idx}_x'][:]       # (H, W, 3) uint8 BGR
        rgb = bgr[:, :, ::-1].copy()
        return torch.from_numpy(rgb).permute(2, 0, 1)  # (3, H, W) uint8

    def __getitem__(self, idx):
        k        = self._ep_of(idx)
        ep       = self.episodes[k]
        n        = ep['n']
        prev_end = self.ep_ends[k - 1] if k > 0 else 0
        i        = idx - prev_end
        j        = torch.randint(0, n, (1,)).item()

        frame_i   = self._read_frame(ep['path'], i)
        frame_j   = self._read_frame(ep['path'], j)
        delta_xyz = ep['xyz'][j] - ep['xyz'][i]   # (3,) in mm
        return frame_i, frame_j, delta_xyz


dataset     = PairDisplacementDataset(episode_data)
loader      = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)
print(f'Train dataset size: {len(dataset):,} pairs/epoch')

val_dataset = PairDisplacementDataset(val_episode_data)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, drop_last=False)
print(f'Val dataset size  : {len(val_dataset):,} pairs')

## Model

**Encoder (last block trainable)**: `vit_small_plus_patch16_dinov3.lvd1689m` (timm), Meta's
official DINOv3 ViT-S+/16 weights. Natively produces a 384-dim pooled CLS embedding
(`forward_head(..., pre_logits=True)`) — no manual pooling layer of ours is needed.

**Head** (trainable): `[emb_a, emb_b, emb_a − emb_b]` (1152-dim) → small MLP → `(3,)`.

In [ ]:
DINO_MODEL_NAME  = 'vit_small_plus_patch16_dinov3.lvd1689m'
EMB_DIM          = 384   # ViT-S+/16 native embed_dim (pooled/CLS output)
UNFREEZE_BLOCKS  = ('blocks.11', 'norm.')   # only fine-tune the last transformer block + final norm

dino = timm.create_model(DINO_MODEL_NAME, pretrained=True).to(device)

for name, p in dino.named_parameters():
    p.requires_grad_(any(name.startswith(s) for s in UNFREEZE_BLOCKS))
dino.train()   # last block + final norm fine-tuned, rest frozen

_data_cfg = timm.data.resolve_data_config({}, model=dino)
DINO_MEAN = torch.tensor(_data_cfg['mean'], device=device).view(1, 3, 1, 1)
DINO_STD  = torch.tensor(_data_cfg['std'],  device=device).view(1, 3, 1, 1)

n_total     = sum(p.numel() for p in dino.parameters())
n_trainable = sum(p.numel() for p in dino.parameters() if p.requires_grad)
print(f'DINO params: {n_total:,} total, {n_trainable:,} trainable ({UNFREEZE_BLOCKS})')


def to_float(x):
    """(B, 3, H, W) uint8 -> (B, 3, H, W) float32 [0,1] on device."""
    return x.to(device, dtype=torch.float32).div_(255.0)


def extract_embedding(x_float):
    """(B, 3, 224, 224) float32 [0,1] -> (B, EMB_DIM) native pooled CLS embedding.

    NOT wrapped in @torch.no_grad() — the unfrozen block(s) need gradients to flow through.
    The frozen blocks run un-detached too (cheap forward pass); autograd just won't build a
    graph past their outputs since their params have requires_grad=False.
    """
    x = (x_float - DINO_MEAN) / DINO_STD
    feat = dino.forward_features(x)              # (B, 1+n_storage+n_patches, EMB_DIM)
    return dino.forward_head(feat, pre_logits=True)   # (B, EMB_DIM) pooled CLS embedding


class CLSDisplacementHead(nn.Module):
    """[emb_a, emb_b, emb_a - emb_b] -> small MLP -> (3,)."""

    def __init__(self, emb_dim=EMB_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(3 * emb_dim),
            nn.Linear(3 * emb_dim, 512),
            nn.SiLU(),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Linear(256, 3),
        )

    def forward(self, emb_a, emb_b):
        return self.net(torch.cat([emb_a, emb_b, emb_a - emb_b], dim=-1))


def sigreg_loss(emb, gamma=1.0, eps=1e-4):
    """Variance/'sigreg' hinge loss: penalize embedding dimensions whose batch-wise std
    falls below gamma, to prevent representation collapse while fine-tuning the encoder.
    """
    std = torch.sqrt(emb.var(dim=0) + eps)
    return F.relu(gamma - std).mean()


head = CLSDisplacementHead(emb_dim=EMB_DIM).to(device)
print(f'Head params (trainable): {sum(p.numel() for p in head.parameters()):,}')

with torch.no_grad():
    _x    = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _emb  = extract_embedding(_x)
    _out  = head(_emb[:1], _emb[1:])
    print(f'Embedding shape : {_emb.shape}')   # expect (2, 384)
    print(f'Output shape    : {_out.shape}')   # expect (1, 3)

In [ ]:
# Compute per-axis mean and std from random pairs — same distribution as training.
_rng = torch.Generator()
_rng.manual_seed(0)
_sample_deltas = []
for ep in episode_data:
    xyz = ep['xyz']    # (n, 3)
    n   = xyz.shape[0]
    i_idx = torch.randint(0, n, (2000,), generator=_rng)
    j_idx = torch.randint(0, n, (2000,), generator=_rng)
    _sample_deltas.append(xyz[j_idx] - xyz[i_idx])

_sample_deltas = torch.cat(_sample_deltas, dim=0)   # (N, 3)
delta_mean = _sample_deltas.mean(0).to(device)
delta_std  = _sample_deltas.std(0).clamp(min=1e-3).to(device)

AXIS_NAMES = ['x', 'y', 'z']
print('delta_xyz stats from random pairs (mm):')
for i, name in enumerate(AXIS_NAMES):
    print(f'  {name}: mean={delta_mean[i]:.3f}  std={delta_std[i]:.3f}')

## Training
Only `head.parameters()` are optimized — the DINO encoder stays frozen and is always run under `torch.no_grad()`.

In [ ]:
STARTING_EPOCH = 0
NUM_EPOCHS     = 50
SIGREG_WEIGHT  = 0.05
SIGREG_GAMMA   = 1.0

dino_trainable_params = [p for p in dino.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW([
    {'params': dino_trainable_params, 'lr': 1e-5},   # small LR: don't destroy the pretrained encoder
    {'params': head.parameters(), 'lr': 1e-3},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


def run_validation():
    dino.eval()
    head.eval()
    losses, abs_errors = [], []
    with torch.no_grad():
        for frame_i, frame_j, delta_xyz in val_loader:
            frame_i   = to_float(frame_i)
            frame_j   = to_float(frame_j)
            delta_xyz = delta_xyz.to(device)
            delta_norm = (delta_xyz - delta_mean) / delta_std

            emb_i = extract_embedding(frame_i)
            emb_j = extract_embedding(frame_j)
            pred  = head(emb_i, emb_j)
            loss  = F.huber_loss(pred, delta_norm, delta=1.0)

            pred_mm = pred * delta_std + delta_mean
            abs_errors.append((pred_mm - delta_xyz).abs().mean().item())
            losses.append(loss.item())
    dino.train()
    head.train()
    return float(np.mean(losses)), float(np.mean(abs_errors))


dino.train()
head.train()

for epoch in range(STARTING_EPOCH, NUM_EPOCHS):
    losses, abs_errors, sigreg_losses = [], [], []

    for frame_i, frame_j, delta_xyz in tqdm(loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        frame_i   = to_float(frame_i)
        frame_j   = to_float(frame_j)
        delta_xyz = delta_xyz.to(device)
        delta_norm = (delta_xyz - delta_mean) / delta_std

        emb_i = extract_embedding(frame_i)
        emb_j = extract_embedding(frame_j)

        pred = head(emb_i, emb_j)                              # (B, 3) normalised
        huber = F.huber_loss(pred, delta_norm, delta=1.0)
        sreg  = sigreg_loss(torch.cat([emb_i, emb_j], dim=0), gamma=SIGREG_GAMMA)
        loss  = huber + SIGREG_WEIGHT * sreg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dino_trainable_params + list(head.parameters()), max_norm=5.0)
        optimizer.step()

        pred_mm = pred.detach() * delta_std + delta_mean
        abs_errors.append((pred_mm - delta_xyz).abs().mean().item())
        losses.append(huber.item())
        sigreg_losses.append(sreg.item())

    scheduler.step()
    val_loss, val_abs_err = run_validation()
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
          f'train_loss={np.mean(losses):.4f}  sigreg={np.mean(sigreg_losses):.4f}  '
          f'train_mean|err|={np.mean(abs_errors):.3f} mm  '
          f'val_loss={val_loss:.4f}  val_mean|err|={val_abs_err:.3f} mm  '
          f'lr={scheduler.get_last_lr()[-1]:.2e}')

    if (epoch + 1) % 10 == 0 or epoch + 1 == NUM_EPOCHS:
        torch.save({
            'epoch': epoch + 1,
            'dino': dino.state_dict(),
            'head': head.state_dict(),
            'delta_mean': delta_mean.cpu(),
            'delta_std': delta_std.cpu(),
        }, CKPT_DIR / f'dino_disp_epoch_{epoch+1:04d}.pt')

## Evaluation
### Load checkpoint

In [ ]:
ckpts = sorted(CKPT_DIR.glob('dino_disp_epoch_*.pt'))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location=device, weights_only=False)
    dino.load_state_dict(ckpt['dino'])
    head.load_state_dict(ckpt['head'])
    delta_mean = ckpt['delta_mean'].to(device)
    delta_std  = ckpt['delta_std'].to(device)
    print(f'Loaded: {ckpts[-1].name}  (epoch {ckpt["epoch"]})')
else:
    print('No checkpoints — run training first.')

dino.eval()
head.eval()

### Per-axis error distribution

In [ ]:
all_preds, all_gt = [], []

with torch.no_grad():
    for frame_i, frame_j, delta_xyz in tqdm(val_loader, desc='Evaluating'):
        frame_i = to_float(frame_i.to(device))
        frame_j = to_float(frame_j.to(device))
        emb_i = extract_embedding(frame_i)
        emb_j = extract_embedding(frame_j)
        pred_norm = head(emb_i, emb_j)
        pred_mm   = (pred_norm * delta_std + delta_mean).cpu()
        all_preds.append(pred_mm)
        all_gt.append(delta_xyz)

all_preds = torch.cat(all_preds).numpy()
all_gt    = torch.cat(all_gt).numpy()
errors    = all_preds - all_gt

print('Per-axis MAE (mm) — held-out validation file:')
for i, name in enumerate(AXIS_NAMES):
    mae  = np.abs(errors[:, i]).mean()
    rmse = np.sqrt((errors[:, i] ** 2).mean())
    print(f'  {name}: MAE={mae:.3f} mm   RMSE={rmse:.3f} mm')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, name in enumerate(AXIS_NAMES):
    ax  = axes[i]
    gt  = all_gt[:, i]
    pr  = all_preds[:, i]
    lim = max(np.abs(gt).max(), np.abs(pr).max()) * 1.05
    ax.scatter(gt, pr, s=2, alpha=0.3, color='steelblue', rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], 'r--', lw=1.2, label='ideal')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel(f'GT Δ{name} (mm)'); ax.set_ylabel(f'Pred Δ{name} (mm)')
    ax.set_title(f'Δ{name}  MAE={np.abs(errors[:, i]).mean():.3f} mm')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Qualitative: displacement vs. a fixed reference frame, over time
One cell on a **training** episode, one on the **held-out validation** episode — same fixed-reference
comparison, so you can directly compare how well the trend is tracked in each.

In [ ]:
Q_EPISODE_DATA = episode_data   # TRAIN episode
EPISODE_IDX    = 0              # index into Q_EPISODE_DATA
REF_FRAME_IDX  = 0              # fixed reference frame; every step compares frame t back to this one
N_STEPS        = 80

ep = Q_EPISODE_DATA[EPISODE_IDX]
T  = min(N_STEPS, ep['n'] - 1)


def read(idx):
    with h5py.File(ep['path'], 'r') as f:
        bgr = f[f'frame_{idx}_x'][:]
    rgb = bgr[:, :, ::-1].copy()
    return to_float(torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device))


all_preds_seq, all_gt_seq = [], []

with torch.no_grad():
    f_ref   = read(REF_FRAME_IDX)
    emb_ref = extract_embedding(f_ref)
    for t in tqdm(range(1, T + 1), desc='Sequence eval'):
        ft      = read(t)
        emb_t   = extract_embedding(ft)
        pred_mm = (head(emb_ref, emb_t) * delta_std + delta_mean).squeeze(0).cpu().numpy()
        gt      = (ep['xyz'][t] - ep['xyz'][REF_FRAME_IDX]).numpy()
        all_preds_seq.append(pred_mm)
        all_gt_seq.append(gt)

all_preds_seq = np.stack(all_preds_seq)
all_gt_seq    = np.stack(all_gt_seq)

print('Step-wise MAE per axis (mm) — training file:')
for i, name in enumerate(AXIS_NAMES):
    mae = np.abs(all_preds_seq[:, i] - all_gt_seq[:, i]).mean()
    print(f'  {name}: {mae:.3f} mm')

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for i, name in enumerate(AXIS_NAMES):
    axes[i].plot(range(1, T + 1), all_gt_seq[:, i],    label='GT',   lw=1.5)
    axes[i].plot(range(1, T + 1), all_preds_seq[:, i], label='Pred', lw=1.5, linestyle='--')
    axes[i].set_ylabel(f'Δ{name} (mm)')
    axes[i].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('frame t (compared against fixed reference frame)')
plt.suptitle(f'[TRAIN] Displacement vs. fixed reference frame {REF_FRAME_IDX} — episode {EPISODE_IDX}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
Q_EPISODE_DATA = val_episode_data   # VALIDATION (held-out) episode
EPISODE_IDX    = 0                  # index into Q_EPISODE_DATA
REF_FRAME_IDX  = 0                  # fixed reference frame; every step compares frame t back to this one
N_STEPS        = 80

ep = Q_EPISODE_DATA[EPISODE_IDX]
T  = min(N_STEPS, ep['n'] - 1)


def read(idx):
    with h5py.File(ep['path'], 'r') as f:
        bgr = f[f'frame_{idx}_x'][:]
    rgb = bgr[:, :, ::-1].copy()
    return to_float(torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device))


all_preds_seq, all_gt_seq = [], []

with torch.no_grad():
    f_ref   = read(REF_FRAME_IDX)
    emb_ref = extract_embedding(f_ref)
    for t in tqdm(range(1, T + 1), desc='Sequence eval'):
        ft      = read(t)
        emb_t   = extract_embedding(ft)
        pred_mm = (head(emb_ref, emb_t) * delta_std + delta_mean).squeeze(0).cpu().numpy()
        gt      = (ep['xyz'][t] - ep['xyz'][REF_FRAME_IDX]).numpy()
        all_preds_seq.append(pred_mm)
        all_gt_seq.append(gt)

all_preds_seq = np.stack(all_preds_seq)
all_gt_seq    = np.stack(all_gt_seq)

print('Step-wise MAE per axis (mm) — held-out validation file:')
for i, name in enumerate(AXIS_NAMES):
    mae = np.abs(all_preds_seq[:, i] - all_gt_seq[:, i]).mean()
    print(f'  {name}: {mae:.3f} mm')

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for i, name in enumerate(AXIS_NAMES):
    axes[i].plot(range(1, T + 1), all_gt_seq[:, i],    label='GT',   lw=1.5)
    axes[i].plot(range(1, T + 1), all_preds_seq[:, i], label='Pred', lw=1.5, linestyle='--')
    axes[i].set_ylabel(f'Δ{name} (mm)')
    axes[i].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('frame t (compared against fixed reference frame)')
plt.suptitle(f'[VAL] Displacement vs. fixed reference frame {REF_FRAME_IDX} — held-out episode {EPISODE_IDX}', fontsize=11)
plt.tight_layout()
plt.show()